# eg8 (v2) — layered reveal: a GeoGebra slider on the applet + pull (`listen` / `wait_update`), no ipywidgets

v1 (`eg8_widgets.ipynb`) loaded a `.ggb`, parsed the construction into a DAG with `ConstructionTreeParser` (regexes over command strings), and drove `setLayerVisible` / `setVisible` from an `ipywidgets.IntSlider` watched by an asyncio task.
v2 (rulings 2026-09-03 / 09-09): ipywidgets are not relied on → the slider lives in the applet; the kernel reacts by pull (`wait_update`, no thread); the DAG is the XML's own `<command>` graph (`command_edges`); visibility is changed with GeoGebra scripting commands (`HideLayer` / `ShowLayer`) through the `eval` verb — no new host verb.

In [ ]:
import sys, time; sys.path.insert(0, '/Users/manabu/work/ggblab-replay')
import polars as pl, networkx as nx
from ggblab_extra import ConstructionIO, read_ggb, command_edges
import ggblab.host.html_host as H; H.DEPLOY = 'https://cdn.geogebra.org/apps/deployggb.js'
from ggblab import GeoGebra

In [ ]:
g = GeoGebra(appName='suite', showToolBar=True, showAlgebraInput=True); g

## 1. load the construction: `xml_in` (setXML) once, `xml_out` once → DataFrame (data-in)

In [ ]:
xml0 = read_ggb('2025_06_08.ggb'); t0 = time.time(); r = g.set_xml(xml0, timeout=60); xml = g.xml(timeout=60)
df = ConstructionIO.from_xml(xml); print('ROWS', df.height, 'in', round(time.time() - t0, 2), 's')
layers = df.group_by('Layer').len().sort('Layer'); print(layers.to_pandas().to_string(index=False))

## 2. the DAG is in the XML (`<command>` inputs → outputs); v1's regex parser is retired

In [ ]:
edges = command_edges(xml); G = nx.DiGraph(edges)
print('nodes', G.number_of_nodes(), 'edges', G.number_of_edges(), 'roots', [n for n in G if G.in_degree(n) == 0], 'DAG', nx.is_directed_acyclic_graph(G))
l1 = df.filter(pl.col('Layer').is_in([9, 0, 1]))['Name'].to_list()
nx.write_network_text(G.subgraph(l1))

## 3. hide layers 1..9 with scripting commands in one `eval`; verify by reading the XML back (`ShowObject` per layer)

In [ ]:
def visible_per_layer():
    d = ConstructionIO.from_xml(g.xml(timeout=60))
    return {int(k): int(v) for k, v in d.group_by('Layer').agg(pl.col('ShowObject').sum()).sort('Layer').iter_rows()}
print('before', visible_per_layer())
r = g.command(*[f'HideLayer({k})' for k in range(1, 10)], timeout=30); print('HideLayer x9 ->', r)
print('after ', visible_per_layer())

## 4. the slider is a GeoGebra object; the kernel waits for its updates (pull; drain the backlog first)

A change of the slider is reacted to once per value: GeoGebra fires its update listener several times for one `SetValue` (measured below), and `wait_update` returns every pending event without skipping (ruling 09-09), so the loop compares the value it read.

In [ ]:
print(g.command('n = Slider(0, 9, 1)'), 'n =', g.value('n')); _ = g.events()   # drain the backlog before the loop (ruling 09-09)
last, steps, t_all = None, 0, time.time()
while steps < 3 and time.time() - t_all < 90:
    t0 = time.time(); e = g.wait_update('n', timeout=30)
    if e is None: print('  no change within 30 s'); break
    n = int(round(float(g.value('n'))))
    if n == last: print(f'  (another update:n event for the same value {n} after {round(time.time() - t0, 2)} s — one SetValue fires several)'); continue
    r = g.command(*[(f'ShowLayer({k})' if k <= n else f'HideLayer({k})') for k in range(1, 10)], timeout=30)
    vis = visible_per_layer(); last = n; steps += 1
    print(f'step {steps}: woke after {round(time.time() - t0, 2)} s: {e} -> n = {n}; visible per layer {vis}')

In [ ]:
print('DONE')